# Project 3 — Nhiệm vụ 1: Truy vấn đa phương thức & đánh giá chuẩn

**Nhánh A — text → image** trên **Visual Genome ∩ MS-COCO** (ảnh VG có `coco_id`; caption COCO = truy vấn; scene graph VG = phía đồ thị cho Huy/Dương).

Notebook này tạo ra mọi thứ mà các thành viên khác cắm vào:

| Đầu việc (bảng phân công) | Ô | Đầu ra (trong `WORK/`) |
|---|---|---|
| 1. Trích đặc trưng ảnh + caption bằng **OpenCLIP đóng băng**, lưu lại; **FAISS** top-k theo cosine | §3, §4 | `features/clip_*.npz`, `features/faiss_*.index`, `candidates/*.npz` |
| 2. Metric **Recall@1/5/10, MRR, nDCG@10** (+ MedR) | §2 | `mmxai_eval.py` |
| 3. **Baseline CLIP thuần**, ≥3 seed, mean ± std | §5 | `results/clip_baseline.json`, `results/clip_baseline.md` |
| 4. **Module đánh giá dùng chung** — mọi mô hình chạy cùng split, cùng gallery, cùng top-K | §2, §7 | `mmxai_eval.py`, `split/split.json` |
| 5. Mở rộng: **image → text** (nhánh B), **đánh giá theo nhóm truy vấn** trực tiếp / gián tiếp | §5, §6 | trong `results/` |

**Cài đặt Kaggle:** Accelerator *GPU T4 x1* (CPU cũng chạy được, chậm hơn ~10×) · **Internet On** (tải metadata VG/COCO và ảnh VG qua URL) ·
*Add Input* → dataset `yenphamnhu/visual-genome` (chỉ chứa `scene_graphs.json`).
Mỗi bước có cache trong `WORK/`; chạy lại notebook sẽ bỏ qua bước đã xong.

**Dữ liệu dùng & nguồn**
- `scene_graphs.json` (Kaggle `yenphamnhu/visual-genome`, bản VG 1.4): đối tượng + quan hệ từng ảnh.
- `image_data.json` (VG, 1.7 MB): `coco_id`, URL ảnh. Ảnh được **stream qua URL `cs.stanford.edu`**, không lưu xuống đĩa (vừa quota Kaggle).
- `dataset_coco.json` (Karpathy split, 36 MB): 5 caption/ảnh + split chuẩn train / restval / val / test.

## 1. Cấu hình

`SMOKE=1` (biến môi trường `MMXAI_SMOKE`) chạy thử toàn bộ pipeline với vài trăm ảnh trong ~5 phút.
Nếu Yến đã chốt file split thì điền `split_file` — notebook sẽ dùng đúng file đó thay vì tự tạo.

In [ ]:
import os, sys, json, glob, time, zipfile
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
SMOKE = bool(int(os.environ.get("MMXAI_SMOKE", "0")))

CFG = dict(
    scene_graphs=None,                     # None = tự tìm scene_graphs.json trong /kaggle/input hoặc RAW
    work=os.environ.get("MMXAI_WORK", "/kaggle/working/task1" if ON_KAGGLE else "./work_task1"),
    raw=os.environ.get("MMXAI_RAW", "/kaggle/working/raw" if ON_KAGGLE else "./raw"),
    split_file=None,                       # file split chung của nhóm (json {"train":[coco_id],"val":[...],"test":[...]}); None = tự tạo
    seed_data=0,                           # seed cố định cho việc tạo split / gallery train
    n_train=15000, n_val=None, n_test=None,  # None = toàn bộ pool Karpathy ∩ VG (val 2126, test 2138)
    seeds=[0, 1, 2],                       # seed cho fold gallery 1K (và cho khởi tạo mô hình của Huy/Khánh)
    gallery_size=1000, top_k=100,
    clip_model="ViT-B-32", clip_pretrained="laion2b_s34b_b79k",
    batch_size=256, workers=32, shard_size=2000,
    captions_per_image_train=2,            # số caption/ảnh train đưa vào candidates (hard negative cho GNN)
)
if SMOKE:
    CFG.update(n_train=300, n_val=120, n_test=150, gallery_size=100, top_k=20, workers=8, shard_size=100, batch_size=32)

WORK, RAW = Path(CFG["work"]), Path(CFG["raw"])
for d in (WORK, RAW, WORK / "data", WORK / "split", WORK / "features", WORK / "candidates", WORK / "results"):
    d.mkdir(parents=True, exist_ok=True)
print("Kaggle:", ON_KAGGLE, "| SMOKE:", SMOKE, "| WORK:", WORK.resolve())
print(json.dumps(CFG, indent=1))

In [ ]:
# Thư viện thiếu trên Kaggle: open_clip, faiss, ijson (đọc JSON 739 MB theo luồng, tiết kiệm RAM)
import importlib, subprocess
missing = [p for p, m in [("open_clip_torch", "open_clip"), ("faiss-cpu", "faiss"), ("ijson", "ijson")]
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import numpy as np, pandas as pd, torch, open_clip, faiss, ijson, requests
from PIL import Image
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "",
      "| open_clip", open_clip.__version__, "| faiss", faiss.__version__, "| ijson backend", ijson.backend)

## 2. Module đánh giá dùng chung `mmxai_eval.py`

Ô dưới ghi module ra file để **mọi thành viên import cùng một bản** (`from mmxai_eval import Protocol, fuse, tune_alpha, ...`).
Nội dung chính:

- **Metric** (`rank_metrics`): Recall@1/5/10, MRR, nDCG@10, MedR — tính từ *hạng của ảnh đúng*; hoà điểm xếp ảnh đúng trước.
  `multi_relevant_metrics` cho bộ truy vấn có nhiều ảnh liên quan (bộ ≥20 truy vấn của Khánh).
- **Giao thức** (`Protocol`): `candidates(split, seed)` → top-K ứng viên CLIP (cùng một bộ cho mọi phương pháp);
  `evaluate(score_fn, ...)` → chạy trên 3 fold gallery 1K theo seed + toàn bộ pool, gộp **mean ± std**, tách **nhóm truy vấn**.
  `evaluate_i2t` cho chiều image → text.
- **Kết hợp điểm** (`fuse`, `tune_alpha`): `score = α·z(cos) + (1−α)·z(s_G)` với z-score trong top-K; α **chỉ chọn trên val**.
- **Kiểm định** (`paired_bootstrap`): hiệu MRR/R@1 giữa 2 phương pháp trên cùng truy vấn + khoảng tin cậy 95 %.

In [ ]:
%%writefile mmxai_eval.py
"""mmxai_eval — module đánh giá dùng chung cho Project 3 (Nhiệm vụ 1: truy vấn & đánh giá chuẩn).

Mọi phương pháp (CLIP thuần, CLIP + Graph của Huy, phương pháp mới của Khánh) đều đi qua CÙNG MỘT
giao thức: cùng file split, cùng gallery / fold theo seed, cùng top-K ứng viên CLIP, cùng hàm metric.
Một phương pháp chỉ cần cung cấp hàm  ``score_fn(cands, protocol) -> np.ndarray (Nq, K)``:
điểm cuối cùng của từng ứng viên trong top-K CLIP (điểm cao = xếp trước).

Ví dụ:
    P = Protocol.load("work_task1")                          # split + đặc trưng CLIP + nhóm truy vấn
    clip = lambda c, p: c["sim"]                              # baseline CLIP: điểm = cosine
    res = P.evaluate(clip, name="CLIP", split="test", seeds=[0, 1, 2])
    print(summary_table([res]))

    def clip_graph(c, p):                                     # CLIP + Graph (Huy)
        sg = my_graph_score(c["q_cap"], c["cand"])            # (Nq, K) điểm đồ thị
        return fuse(c["sim"], sg, alpha=0.7)                  # α·z(cos) + (1−α)·z(s_G)
    res2 = P.evaluate(clip_graph, name="CLIP+GAT", split="test", seeds=[0, 1, 2])
    print(paired_bootstrap(res["per_seed"][0]["ranks"], res2["per_seed"][0]["ranks"]))

Thành phần:
    * chuẩn hoá văn bản không phụ thuộc thư viện ngoài (``norm_phrase``, ``ConceptLinker``)
    * metric xếp hạng: Recall@k, MRR, MedR, nDCG@10 (1 ảnh đúng) và bản nhiều ảnh liên quan
    * ``Protocol``: gallery / fold theo seed, top-K ứng viên (FAISS nếu có), đánh giá text→image và
      image→text, tách nhóm truy vấn (trực tiếp / gián tiếp)
    * ``fuse`` / ``tune_alpha``: kết hợp điểm α·z(cos) + (1−α)·z(s_G), chọn α trên validation
    * ``aggregate`` / ``summary_table`` / ``paired_bootstrap``: mean ± std theo seed, bảng Markdown, kiểm định
"""
from __future__ import annotations

import json
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np

__all__ = [
    "rank_metrics", "multi_relevant_metrics", "ranks_from_scores", "rerank_ranks", "zscore", "fuse",
    "tune_alpha", "aggregate", "summary_table", "paired_bootstrap", "Protocol", "ConceptLinker",
    "norm_phrase", "label_query_groups", "topk", "log", "save_json", "load_json",
]

METRICS = ["R@1", "R@5", "R@10", "MRR", "nDCG@10"]
_T0 = time.time()


def log(msg):
    print(f"[{time.time() - _T0:7.1f}s] {msg}", flush=True)


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    def default(o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        raise TypeError(type(o))

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=default)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# =============================================================================== text normalisation
STOPWORDS = set("""
a an the and or but if then than so of at by for with about against between into through during before
after above below to from up down in out on off over under again further once here there when where why how
all any both each few more most other some such no nor not only own same too very s t can will just don
should now is are was were be been being am have has had having do does did doing would could ought i me
my myself we our ours ourselves you your yours yourself yourselves he him his himself she her hers herself
it its itself they them their theirs themselves what which who whom this that these those while as until
one two three four five six seven eight nine ten several many lot lots something someone somebody thing
things another next near around along front top side back also very image picture photo view shot
showing shows show close closeup
""".split())
_TOKEN_RE = re.compile(r"[a-z]+")
_IRREG = {"men": "man", "women": "woman", "children": "child", "people": "people", "feet": "foot",
          "teeth": "tooth", "mice": "mouse", "geese": "goose", "knives": "knife", "leaves": "leaf",
          "shelves": "shelf", "wolves": "wolf", "glasses": "glasses", "jeans": "jeans", "pants": "pants",
          "shorts": "shorts", "scissors": "scissors", "bus": "bus", "grass": "grass", "dress": "dress",
          "skis": "ski", "fries": "fries", "sunglasses": "sunglasses", "clothes": "clothes", "tennis": "tennis",
          "news": "news", "lens": "lens", "gas": "gas", "canvas": "canvas", "atlas": "atlas", "chess": "chess"}


def rule_lemma(w):
    """Lemmatiser luật đơn giản (không cần NLTK): 'dogs'->'dog', 'buses'->'bus', 'riding'->'rid' (chấp nhận)."""
    if w in _IRREG or w.endswith(("us", "is", "ss")):
        return _IRREG.get(w, w)
    if len(w) > 4 and w.endswith("ies"):
        return w[:-3] + "y"
    if len(w) > 4 and w.endswith("es") and w[-3] in "sxzh":
        return w[:-2]
    if len(w) > 3 and w.endswith("s") and not w.endswith("ss"):
        return w[:-1]
    return w


def tokenize(text):
    return _TOKEN_RE.findall(text.lower())


def norm_phrase(text, max_words=3):
    """'Tennis Rackets' -> 'tennis racket'. Trả về None nếu rỗng hoặc quá dài."""
    toks = [rule_lemma(t) for t in tokenize(str(text).replace("_", " "))]
    toks = [t for t in toks if t and t not in STOPWORDS] or toks
    if not toks or len(toks) > max_words:
        return None
    return " ".join(toks)


class ConceptLinker:
    """Khớp tham lam n-gram (dài nhất trước) của truy vấn với từ vựng khái niệm (tên đối tượng VG)."""

    def __init__(self, vocab, max_ngram=3):
        self.vocab = set(vocab)
        self.max_ngram = max_ngram

    def link(self, text):
        seq = [rule_lemma(t) for t in tokenize(text)]
        out, i = [], 0
        while i < len(seq):
            hit = None
            for n in range(min(self.max_ngram, len(seq) - i), 0, -1):
                gram = seq[i:i + n]
                if gram[0] in STOPWORDS or gram[-1] in STOPWORDS:
                    continue
                name = " ".join(gram)
                if name in self.vocab:
                    hit = (name, n)
                    break
            if hit is None:
                i += 1
                continue
            if hit[0] not in out:
                out.append(hit[0])
            i += hit[1]
        return out


# Màu sắc / thuộc tính / danh từ chung chung xuất hiện như "đối tượng" trong VG nhưng không phải thực thể thị giác —
# bỏ khỏi việc GÁN NHÓM truy vấn (đồ thị của Huy vẫn có thể dùng chúng).
GENERIC_CONCEPTS = set("""
white black blue green brown yellow orange red pink gray grey purple gold silver tan dark light color colour
group bunch area piece background foreground day night air body row part side top front middle number couple pair
lot variety kind type set picture image photo scene view stand spot line edge corner shape pattern design
""".split())


def label_query_groups(images, captions, vocab, generic=GENERIC_CONCEPTS):
    """Nhóm truy vấn theo scene graph của ảnh đúng (chỉ để BÁO CÁO, mô hình không dùng).

    direct     : mọi khái niệm liên kết được từ caption đều có trong objects của ảnh đúng
    indirect   : có ít nhất 1 khái niệm KHÔNG có trong ảnh (chỉ nối được qua quan hệ / đồng nghĩa)
    no_concept : caption không liên kết được khái niệm nào trong từ vựng
    """
    linker = ConceptLinker(vocab)
    groups, concepts = [], []
    for c in captions:
        cs = [x for x in linker.link(c["text"]) if x not in generic]
        objs = set(images[c["img"]]["objects"])
        concepts.append(cs)
        if not cs:
            groups.append("no_concept")
        elif all(x in objs for x in cs):
            groups.append("direct")
        else:
            groups.append("indirect")
    return np.array(groups), concepts


# =============================================================================== metrics
def rank_metrics(ranks, ks=(1, 5, 10)):
    """Metric khi mỗi truy vấn có đúng 1 ảnh đúng. ``ranks``: hạng (1-based) của ảnh đúng."""
    r = np.asarray(ranks, dtype=np.float64)
    m = {f"R@{k}": float((r <= k).mean() * 100) for k in ks}
    m["MRR"] = float((1.0 / r).mean() * 100)
    m["nDCG@10"] = float(np.where(r <= 10, 1.0 / np.log2(r + 1), 0.0).mean() * 100)
    m["MedR"] = float(np.median(r))
    m["n"] = int(len(r))
    return m


def multi_relevant_metrics(order, relevant, ks=(1, 5, 10)):
    """Metric khi một truy vấn có NHIỀU ảnh liên quan (bộ ≥20 truy vấn của Khánh).

    ``order``: danh sách id ảnh đã xếp hạng; ``relevant``: tập id ảnh liên quan.
    """
    hits = np.array([i in relevant for i in order], dtype=float)
    first = int(np.argmax(hits)) + 1 if hits.any() else len(order) + 1
    m = {f"R@{k}": float(hits[:k].any() * 100) for k in ks}
    m["MRR"] = 100.0 / first
    m["P@10"] = float(hits[:10].mean() * 100)
    disc = 1.0 / np.log2(np.arange(2, 12))
    ideal = disc[: min(len(relevant), 10)].sum()
    m["nDCG@10"] = float((hits[:10] * disc[: len(hits[:10])]).sum() / ideal * 100) if ideal else 0.0
    return m


def ranks_from_scores(scores, gt_col):
    """Hạng (1-based) của cột gt_col trong từng hàng điểm. Hoà điểm: xếp ảnh đúng trước (đếm 'lớn hơn hẳn')."""
    gt = scores[np.arange(len(scores)), gt_col][:, None]
    return (scores > gt).sum(1) + 1


def rerank_ranks(cands, scores):
    """Hạng ảnh đúng sau khi xếp lại top-K bằng ``scores`` (Nq, K); ảnh đúng ngoài top-K giữ hạng CLIP."""
    pos = cands["pos"]
    order = np.argsort(-scores, axis=1, kind="stable")
    new_pos = np.argmax(order == pos[:, None], axis=1)
    return np.where(pos >= 0, new_pos + 1, cands["gt_rank"])


def zscore(x):
    mu, sd = x.mean(1, keepdims=True), x.std(1, keepdims=True)
    return np.where(sd > 1e-8, (x - mu) / np.maximum(sd, 1e-8), 0.0)


def fuse(sim, graph_score, alpha):
    """score = α·z(cos) + (1−α)·z(s_G), chuẩn hoá z-score trong top-K của từng truy vấn."""
    return alpha * zscore(np.asarray(sim, np.float64)) + (1 - alpha) * zscore(np.asarray(graph_score, np.float64))


def tune_alpha(cands, graph_score, step=0.05, key="MRR"):
    """Quét α ∈ [0,1] trên một split (PHẢI là validation). Trả về (α tốt nhất, đường cong)."""
    curve = []
    for a in np.round(np.arange(0, 1 + 1e-9, step), 4):
        m = rank_metrics(rerank_ranks(cands, fuse(cands["sim"], graph_score, a)))
        curve.append(dict(alpha=float(a), **m))
    best = max(curve, key=lambda d: (d[key], d["alpha"]))
    return best["alpha"], curve


def aggregate(list_of_metrics, keys=None):
    keys = keys or [k for k in list_of_metrics[0] if isinstance(list_of_metrics[0][k], (int, float)) and k != "n"]
    out = {}
    for k in keys:
        v = np.array([m[k] for m in list_of_metrics], dtype=float)
        out[k] = dict(mean=float(v.mean()), std=float(v.std(ddof=0)), values=v.tolist())
    out["n"] = int(np.mean([m.get("n", 0) for m in list_of_metrics]))
    return out


def _cell(v, digits=2):
    if isinstance(v, dict):
        return f"{v['mean']:.{digits}f} ± {v['std']:.{digits}f}"
    return f"{v:.{digits}f}" if isinstance(v, float) else str(v)


def summary_table(results, metrics=METRICS, field="mean_std", title=None):
    """Bảng Markdown: một dòng / kết quả của ``Protocol.evaluate``."""
    lines = ([f"**{title}**", ""] if title else []) + [
        "| phương pháp | split | n truy vấn | " + " | ".join(metrics) + " |",
        "|---|---|---|" + "---|" * len(metrics)]
    for r in results:
        src = r[field]
        lines.append(f"| {r['name']} | {r['split']} | {src.get('n', '')} | " + " | ".join(_cell(src[m]) for m in metrics) + " |")
    return "\n".join(lines)


def paired_bootstrap(ranks_a, ranks_b, n_boot=2000, seed=0):
    """So sánh 2 phương pháp trên CÙNG các truy vấn (ghép cặp) bằng bootstrap trên hiệu MRR và R@1.

    Trả về hiệu (b − a), khoảng tin cậy 95 % và p-value hai phía ≈ 2·min(P(Δ≤0), P(Δ≥0)).
    """
    a, b = np.asarray(ranks_a, float), np.asarray(ranks_b, float)
    assert a.shape == b.shape, "hai phương pháp phải chạy trên cùng danh sách truy vấn"
    rng = np.random.RandomState(seed)
    out = {}
    for name, fa in (("MRR", lambda r: 100 / r), ("R@1", lambda r: 100.0 * (r <= 1)), ("R@10", lambda r: 100.0 * (r <= 10))):
        d = fa(b) - fa(a)
        idx = rng.randint(0, len(d), size=(n_boot, len(d)))
        boots = d[idx].mean(1)
        p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
        out[name] = dict(delta=float(d.mean()), ci95=[float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],
                         p_value=float(min(1.0, p)))
    return out


# =============================================================================== similarity / top-K
def _device():
    try:
        import torch
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    except Exception:  # noqa: BLE001
        return None


def topk(q, gal, K, use_faiss=True):
    """Top-K theo tích vô hướng (cosine nếu đã chuẩn hoá). Trả về (sim (Nq,K), idx (Nq,K), sims đầy đủ hoặc None)."""
    q = np.ascontiguousarray(q, dtype=np.float32)
    gal = np.ascontiguousarray(gal, dtype=np.float32)
    K = min(K, len(gal))
    if use_faiss:
        try:
            import faiss
            index = faiss.IndexFlatIP(gal.shape[1])
            index.add(gal)
            s, i = index.search(q, K)
            return s, i, index
        except Exception:  # noqa: BLE001
            pass
    sims = q @ gal.T
    i = np.argpartition(-sims, K - 1, axis=1)[:, :K]
    s = np.take_along_axis(sims, i, 1)
    o = np.argsort(-s, axis=1, kind="stable")
    return np.take_along_axis(s, o, 1), np.take_along_axis(i, o, 1), None


def full_sims(q, gal, batch=4096):
    """Ma trận cosine đầy đủ (Nq, Ng), dùng GPU nếu có."""
    dev = _device()
    q = np.asarray(q, np.float32)
    gal = np.asarray(gal, np.float32)
    if dev is not None and dev.type == "cuda":
        import torch
        g = torch.tensor(gal, device=dev)
        out = [(torch.tensor(q[b:b + batch], device=dev) @ g.T).cpu().numpy() for b in range(0, len(q), batch)]
        return np.concatenate(out)
    return q @ gal.T


# =============================================================================== protocol
class Protocol:
    """Giao thức đánh giá chung.

    Dữ liệu (do notebook Nhiệm vụ 1 tạo ra trong ``work_dir``):
        data/images.jsonl   : mỗi dòng 1 ảnh {idx, id, vg_id, coco_id, url, split, objects, relations}
        data/captions.jsonl : mỗi dòng 1 caption {idx, img, text, group}
        split/split.json    : {"train": [...coco_id], "val": [...], "test": [...], "meta": {...}}
        features/clip_<model>.npz : img (Ni,D) float16, cap (Nc,D) float16, cap_img (Nc,), ok (Ni,) bool

    Seed:  CLIP đóng băng là tất định, nên ``seed`` điều khiển (i) fold gallery ``gallery_size`` ảnh lấy
    mẫu từ pool test/val, (ii) thứ tự gallery train. Mô hình học (GNN) dùng cùng seed để khởi tạo.
    ``seed=None`` → dùng toàn bộ pool (kết quả tất định, dùng làm số chính).
    """

    def __init__(self, images, captions, split, feats, gallery_size=1000, K=100, seed_data=0):
        self.images, self.captions, self.split = images, captions, split
        self.img = np.asarray(feats["img"], np.float32)
        self.cap = np.asarray(feats["cap"], np.float32)
        self.cap_img = np.asarray(feats["cap_img"])
        self.ok = np.asarray(feats["ok"], bool)
        self.G, self.K, self.seed_data = gallery_size, K, seed_data
        self.caps_of = {}
        for ci, ii in enumerate(self.cap_img):
            self.caps_of.setdefault(int(ii), []).append(ci)
        self.groups = np.array([c.get("group", "all") for c in captions])
        self._cache = {}

    # ---- loading
    @classmethod
    def load(cls, work_dir, feature_file=None, **kw):
        work = Path(work_dir)
        images = [json.loads(l) for l in open(work / "data" / "images.jsonl", encoding="utf-8")]
        captions = [json.loads(l) for l in open(work / "data" / "captions.jsonl", encoding="utf-8")]
        split = load_json(work / "split" / "split.json")
        feature_file = feature_file or sorted((work / "features").glob("clip_*.npz"))[0]
        feats = np.load(feature_file, allow_pickle=False)
        meta = split.get("meta", {})
        kw.setdefault("gallery_size", meta.get("gallery_size", 1000))
        kw.setdefault("K", meta.get("top_k", 100))
        kw.setdefault("seed_data", meta.get("seed_data", 0))
        log(f"Protocol: {len(images)} ảnh, {len(captions)} caption, features {Path(feature_file).name} "
            f"{feats['img'].shape}, gallery={kw['gallery_size']}, K={kw['K']}")
        return cls(images, captions, split, feats, **kw)

    # ---- galleries
    def pool(self, split):
        return [im["idx"] for im in self.images if im["split"] == split and self.ok[im["idx"]]]

    def gallery(self, split, seed=None):
        pool = self.pool(split)
        if seed is None or len(pool) <= self.G:
            return sorted(pool)
        rng = np.random.RandomState(seed)
        return sorted(rng.choice(pool, self.G, replace=False).tolist())

    def train_galleries(self, seed=None):
        """Train chia thành các gallery ``gallery_size`` ảnh (giống test) để lấy hard negative."""
        pool = self.pool("train")
        rng = np.random.RandomState(self.seed_data if seed is None else seed)
        rng.shuffle(pool)
        return [pool[i:i + self.G] for i in range(0, len(pool), self.G)]

    # ---- candidates
    def candidates(self, split="test", seed=None, galleries=None, ensure_positive=False, captions_per_image=None):
        """Top-K ứng viên CLIP cho mọi caption của ảnh trong gallery (định dạng chung cho mọi phương pháp).

        Trả về dict: q_cap (Nq,) idx caption, q_img (Nq,) idx ảnh đúng, cand (Nq,K) idx ảnh ứng viên,
        sim (Nq,K) cosine, pos (Nq,) cột của ảnh đúng trong cand (−1 nếu ngoài top-K),
        gt_rank (Nq,) hạng CLIP của ảnh đúng trong toàn gallery, gallery (list idx), group (Nq,).
        """
        key = (split, seed, galleries is None, ensure_positive, captions_per_image)
        if key in self._cache:
            return self._cache[key]
        if galleries is None:
            galleries = self.train_galleries(seed) if split == "train" else [self.gallery(split, seed)]
        rng = np.random.RandomState(0 if seed is None else seed)
        out = dict(q_cap=[], q_img=[], cand=[], sim=[], pos=[], gt_rank=[])
        for gal in galleries:
            gal = list(gal)
            col = {g: j for j, g in enumerate(gal)}
            q_cap = []
            for g in gal:
                cs = self.caps_of.get(g, [])
                if captions_per_image and len(cs) > captions_per_image:
                    cs = sorted(rng.choice(cs, captions_per_image, replace=False).tolist())
                q_cap += cs
            q_cap = np.array(q_cap, dtype=np.int64)
            q_img = self.cap_img[q_cap].astype(np.int64)
            sims = full_sims(self.cap[q_cap], self.img[gal])
            gt_col = np.array([col[int(i)] for i in q_img])
            gt_rank = ranks_from_scores(sims, gt_col)
            K = min(self.K, len(gal))
            idx = np.argsort(-sims, axis=1, kind="stable")[:, :K]
            sim = np.take_along_axis(sims, idx, 1).astype(np.float32)
            cand = np.array(gal)[idx]
            hit = cand == q_img[:, None]
            pos = np.where(hit.any(1), hit.argmax(1), -1)
            if ensure_positive:  # train: ảnh đúng luôn nằm trong ứng viên (thay ứng viên cuối)
                miss = pos < 0
                cand[miss, -1] = q_img[miss]
                sim[miss, -1] = sims[miss, gt_col[miss]]
                pos[miss] = K - 1
            for k, v in zip(out, (q_cap, q_img, cand, sim, pos, gt_rank)):
                out[k].append(v)
        res = {k: np.concatenate(v) for k, v in out.items()}
        res["gallery"] = [int(x) for g in galleries for x in g]
        res["gallery_size"] = len(galleries[0])
        res["group"] = self.groups[res["q_cap"]]
        res["split"], res["seed"] = split, seed
        self._cache[key] = res
        return res

    # ---- evaluation text -> image
    def evaluate(self, score_fn, name, split="test", seeds=(0, 1, 2), full=True, by_group=True, verbose=True):
        """Chạy ``score_fn`` trên từng fold theo seed (+ toàn bộ pool nếu ``full``) và gộp mean ± std."""
        per_seed, groups = [], {}
        for seed in list(seeds) + ([None] if full else []):
            c = self.candidates(split, seed)
            scores = np.asarray(score_fn(c, self), dtype=np.float64)
            assert scores.shape == c["sim"].shape, f"score_fn phải trả về (Nq,K)={c['sim'].shape}, nhận {scores.shape}"
            ranks = rerank_ranks(c, scores)
            m = rank_metrics(ranks)
            rec = dict(seed=seed, gallery_size=c["gallery_size"], ranks=ranks.tolist(), **m)
            if by_group:
                rec["by_group"] = {g: rank_metrics(ranks[c["group"] == g]) for g in np.unique(c["group"])}
            per_seed.append(rec)
            if verbose:
                tag = "full" if seed is None else f"seed {seed}"
                log(f"{name:>24s} | {split} {tag:>6s} | gallery {c['gallery_size']:5d} | " +
                    " ".join(f"{k}={m[k]:.2f}" for k in METRICS) + f" MedR={m['MedR']:.0f}")
        seeded = [r for r in per_seed if r["seed"] is not None]
        res = dict(name=name, split=split, seeds=list(seeds), gallery_size=self.G, per_seed=per_seed,
                   mean_std=aggregate(seeded) if seeded else None,
                   full=next((r for r in per_seed if r["seed"] is None), None))
        if by_group and seeded:
            for g in np.unique(self.groups):
                vals = [r["by_group"][g] for r in seeded if g in r["by_group"]]
                if vals:
                    groups[g] = aggregate(vals)
            res["by_group"] = groups
        return res

    # ---- evaluation image -> text (nhánh B)
    def evaluate_i2t(self, name="CLIP", split="test", seeds=(0, 1, 2), full=True, score_fn=None, verbose=True):
        """Truy vấn = ảnh, gallery = mọi caption của ảnh trong gallery; đúng nếu lấy được 1 trong 5 caption.

        ``score_fn(sims, protocol, gallery, cap_idx) -> (Ni, Nc)`` để cắm phương pháp khác (mặc định: cosine).
        """
        per_seed = []
        for seed in list(seeds) + ([None] if full else []):
            gal = self.gallery(split, seed)
            cap_idx = np.array([ci for g in gal for ci in self.caps_of.get(g, [])])
            sims = full_sims(self.img[gal], self.cap[cap_idx])
            if score_fn is not None:
                sims = np.asarray(score_fn(sims, self, gal, cap_idx), np.float64)
            owner = self.cap_img[cap_idx]
            ranks = []
            for r, g in enumerate(gal):
                mine = np.where(owner == g)[0]
                ranks.append(int(ranks_from_scores(sims[r][None, :].repeat(len(mine), 0), mine).min()))
            m = rank_metrics(ranks)
            per_seed.append(dict(seed=seed, gallery_size=len(gal), ranks=ranks, **m))
            if verbose:
                tag = "full" if seed is None else f"seed {seed}"
                log(f"{name + ' (i2t)':>24s} | {split} {tag:>6s} | gallery {len(gal):5d} | " +
                    " ".join(f"{k}={m[k]:.2f}" for k in METRICS) + f" MedR={m['MedR']:.0f}")
        seeded = [r for r in per_seed if r["seed"] is not None]
        return dict(name=name + " (image→text)", split=split, seeds=list(seeds), per_seed=per_seed,
                    mean_std=aggregate(seeded) if seeded else None,
                    full=next((r for r in per_seed if r["seed"] is None), None))

    # ---- helpers for other methods
    def image_objects(self, idx):
        return self.images[int(idx)]["objects"]

    def caption_text(self, idx):
        return self.captions[int(idx)]["text"]

    def object_vocab(self, split="train", min_images=5, max_size=2000):
        """Từ vựng tên đối tượng đếm trên ảnh TRAIN (không rò rỉ từ val/test)."""
        cnt = Counter()
        for im in self.images:
            if im["split"] == split:
                cnt.update(set(im["objects"]))
        return [n for n, c in cnt.most_common(max_size) if c >= min_images]

In [ ]:
import importlib, mmxai_eval
importlib.reload(mmxai_eval)
from mmxai_eval import (Protocol, rank_metrics, multi_relevant_metrics, fuse, tune_alpha, aggregate, summary_table,
                        paired_bootstrap, norm_phrase, label_query_groups, log, save_json, load_json, METRICS)

# kiểm tra nhanh metric bằng ví dụ tay: hạng [1, 2, 11] -> R@1 = 33.3, R@10 = 66.7, MRR = (1 + 1/2 + 1/11)/3
m = rank_metrics([1, 2, 11]); print({k: round(v, 2) for k, v in m.items()})
assert abs(m["R@1"] - 100 / 3) < 1e-6 and abs(m["R@10"] - 200 / 3) < 1e-6 and abs(m["MRR"] - 100 * (1 + .5 + 1 / 11) / 3) < 1e-6
print("multi-relevant:", multi_relevant_metrics(order=[5, 3, 9, 1], relevant={3, 1}))

## 3. Dữ liệu, split cố định và chống rò rỉ

1. **Metadata**: tải `image_data.json` (VG) và `dataset_coco.json` (Karpathy); tìm `scene_graphs.json` trong `/kaggle/input`.
2. **Split** (`split/split.json`): Karpathy `train+restval → train`, `val → val`, `test → test`, giao với VG qua `coco_id`;
   train lấy mẫu `n_train` ảnh với `seed_data`; val/test giữ toàn bộ pool (≈2.1K mỗi tập). File này là **split chung cho cả nhóm**
   (nếu Yến đã chốt file khác → điền `CFG["split_file"]`).
3. **Scene graph**: đọc theo luồng `scene_graphs.json`, chỉ giữ ảnh trong split; chuẩn hoá tên đối tượng / vị từ (chữ thường, bỏ mạo từ, số ít).
4. **Nhóm truy vấn** (chỉ để báo cáo): từ vựng đối tượng đếm **trên train**; caption được gán `direct` nếu mọi khái niệm liên kết được
   đều có trong scene graph của ảnh đúng, `indirect` nếu có khái niệm không có trong ảnh (phải nối qua quan hệ / đồng nghĩa), `no_concept` nếu không liên kết được.
   Màu sắc / thuộc tính / danh từ chung (`white`, `group`, `area`, …) không được tính khi gán nhóm (`GENERIC_CONCEPTS` trong module).
   Scene graph VG không đầy đủ nên nhóm `indirect` thường lớn hơn `direct`; ví dụ caption nói *people* còn VG ghi *man*, *woman*.

**Chống rò rỉ**: caption val/test không tạo bất kỳ thống kê nào (từ vựng chỉ từ train); CLIP đóng băng nên không có tham số học;
α và siêu tham số của các phương pháp khác chỉ chọn trên `val` (xem §7).

In [ ]:
VG_BASE = "https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset"
KARPATHY_URL = "https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip"

def download(url, dest, retries=3):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    for a in range(retries):
        try:
            log(f"tải {url}")
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                tmp = dest.with_suffix(dest.suffix + ".part")
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            os.replace(tmp, dest)
            return dest
        except Exception as e:  # noqa: BLE001
            log(f"  lỗi lần {a + 1}: {e}"); time.sleep(3)
    raise RuntimeError(f"không tải được {url}")

def unzip_member(zip_path, member, out_dir):
    out = Path(out_dir) / Path(member).name
    if not out.exists():
        with zipfile.ZipFile(zip_path) as z:
            name = next(n for n in z.namelist() if n.endswith(member))
            with z.open(name) as src, open(out, "wb") as dst:
                while chunk := src.read(1 << 20):
                    dst.write(chunk)
    return out

def find_scene_graphs():
    cands = [CFG["scene_graphs"]] if CFG["scene_graphs"] else []
    cands += glob.glob("/kaggle/input/**/scene_graphs.json", recursive=True) + [str(RAW / "scene_graphs.json")]
    for c in cands:
        if c and Path(c).exists():
            return Path(c)
    # không có trong input: tải bản gốc từ VG (114 MB zip)
    return unzip_member(download(f"{VG_BASE}/scene_graphs.json.zip", RAW / "scene_graphs.json.zip"), "scene_graphs.json", RAW)

IMAGE_DATA = unzip_member(download(f"{VG_BASE}/image_data.json.zip", RAW / "image_data.json.zip"), "image_data.json", RAW)
COCO_JSON = unzip_member(download(KARPATHY_URL, RAW / "caption_datasets.zip"), "dataset_coco.json", RAW)
SCENE_GRAPHS = find_scene_graphs()
print("scene_graphs.json:", SCENE_GRAPHS, f"{SCENE_GRAPHS.stat().st_size / 1e6:.0f} MB")

In [ ]:
_DROP_PRED = {"a", "an", "the", "be", "is", "are", "s"}

def norm_predicate(p):
    toks = [t for t in mmxai_eval.tokenize(p) if t not in _DROP_PRED]
    return " ".join(toks) if toks else None

def make_split(coco_by_id):
    rng = np.random.RandomState(CFG["seed_data"])
    pools = {"train": [], "val": [], "test": []}
    for cid, im in coco_by_id.items():
        pools["train" if im["split"] in ("train", "restval") else im["split"]].append(cid)
    split = {}
    for s, n in (("train", CFG["n_train"]), ("val", CFG["n_val"]), ("test", CFG["n_test"])):
        ids = sorted(pools[s])
        if n and len(ids) > n:
            ids = sorted(rng.choice(ids, n, replace=False).tolist())
        split[s] = [int(x) for x in ids]
    split["meta"] = dict(source="Karpathy COCO split (train+restval->train, val->val, test->test) ∩ Visual Genome (coco_id)",
                         pool_sizes={k: len(v) for k, v in pools.items()}, seed_data=CFG["seed_data"],
                         n_train=CFG["n_train"], n_val=CFG["n_val"], n_test=CFG["n_test"],
                         gallery_size=CFG["gallery_size"], top_k=CFG["top_k"], seeds=CFG["seeds"],
                         created=time.strftime("%Y-%m-%d %H:%M:%S"), smoke=SMOKE)
    return split

def build_dataset():
    out_img, out_cap, out_split = WORK / "data" / "images.jsonl", WORK / "data" / "captions.jsonl", WORK / "split" / "split.json"
    if out_img.exists() and out_cap.exists() and out_split.exists():
        log("dataset đã có, bỏ qua"); return
    vg_by_coco = {m["coco_id"]: m for m in json.load(open(IMAGE_DATA)) if m.get("coco_id")}
    coco_by_id = {im["cocoid"]: im for im in json.load(open(COCO_JSON))["images"] if im["cocoid"] in vg_by_coco}
    log(f"VG ∩ COCO: {len(coco_by_id)} ảnh")
    if CFG["split_file"]:
        split = load_json(CFG["split_file"]); split.setdefault("meta", {})["source_file"] = CFG["split_file"]
        split["meta"].update(gallery_size=CFG["gallery_size"], top_k=CFG["top_k"], seeds=CFG["seeds"], seed_data=CFG["seed_data"])
        log(f"dùng split của nhóm: {CFG['split_file']}")
    else:
        split = make_split(coco_by_id)
    split_of = {cid: s for s in ("train", "val", "test") for cid in split[s]}
    assert len(split_of) == sum(len(split[s]) for s in ("train", "val", "test")), "split có ảnh trùng giữa các tập!"
    log("split: " + ", ".join(f"{s}={len(split[s])}" for s in ("train", "val", "test")))
    vg2coco = {vg_by_coco[c]["image_id"]: c for c in split_of}

    objs, rels, n_seen = {}, {}, 0
    with open(SCENE_GRAPHS, "rb") as f:
        for rec in ijson.items(f, "item"):
            n_seen += 1
            cid = vg2coco.get(rec["image_id"])
            if cid is None:
                continue
            name_of, cnt = {}, Counter()
            for o in rec.get("objects", []):
                raw = (o.get("names") or [o.get("name")])[0]
                n = norm_phrase(raw) if raw else None
                if n:
                    name_of[o["object_id"]] = n; cnt[n] += 1
            rset = set()
            for r in rec.get("relationships", []):
                s, o, p = name_of.get(r.get("subject_id")), name_of.get(r.get("object_id")), norm_predicate(r.get("predicate", ""))
                if s and o and p and s != o:
                    rset.add((s, p, o))
            objs[cid], rels[cid] = cnt, rset
            if len(objs) == len(split_of):
                break
    log(f"đọc {n_seen} scene graph, giữ {len(objs)} ảnh trong split")

    images, captions = [], []
    for cid in sorted(split_of):
        im, m = coco_by_id[cid], vg_by_coco[cid]
        images.append(dict(idx=len(images), id=f"coco{cid}", coco_id=int(cid), vg_id=int(m["image_id"]), split=split_of[cid],
                           url=m["url"], coco_url=f"http://images.cocodataset.org/{im['filepath']}/{im['filename']}",
                           objects=[n for n, _ in objs.get(cid, Counter()).most_common()],
                           relations=[list(t) for t in sorted(rels.get(cid, ()))]))
        for s in im["sentences"]:
            captions.append(dict(idx=len(captions), img=images[-1]["idx"], text=s["raw"].strip()))
    vocab_cnt = Counter()
    for im in images:
        if im["split"] == "train":
            vocab_cnt.update(set(im["objects"]))
    vocab = [n for n, c in vocab_cnt.most_common(2000) if c >= (2 if SMOKE else 5)]
    groups, concepts = label_query_groups(images, captions, vocab)
    for c, g, cs in zip(captions, groups, concepts):
        c["group"], c["concepts"] = str(g), cs
    with open(out_img, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(im, ensure_ascii=False) + "\n" for im in images)
    with open(out_cap, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(c, ensure_ascii=False) + "\n" for c in captions)
    save_json(split, out_split)
    save_json(vocab, WORK / "data" / "object_vocab_train.json")
    log(f"{len(images)} ảnh, {len(captions)} caption, từ vựng train {len(vocab)} đối tượng; "
        f"TB {np.mean([len(i['objects']) for i in images]):.1f} đối tượng, {np.mean([len(i['relations']) for i in images]):.1f} quan hệ / ảnh")

build_dataset()
images = [json.loads(l) for l in open(WORK / "data" / "images.jsonl", encoding="utf-8")]
captions = [json.loads(l) for l in open(WORK / "data" / "captions.jsonl", encoding="utf-8")]
df_img = pd.DataFrame(images); df_cap = pd.DataFrame(captions)
display(df_img.groupby("split").agg(n_images=("idx", "size"), objects=("objects", lambda s: s.map(len).mean()),
                                    relations=("relations", lambda s: s.map(len).mean())).round(1))
display(pd.crosstab(df_cap.img.map(df_img.split), df_cap.group))
ex = images[0]; print(ex["id"], ex["url"], "\n objects:", ex["objects"][:12], "\n relations:", ex["relations"][:5])
print(" captions:", [c["text"] for c in captions if c["img"] == ex["idx"]][:2], "| group:", captions[0]["group"], captions[0]["concepts"])

## 4. Trích đặc trưng OpenCLIP đóng băng (một lần, lưu lại)

`ViT-B-32 / laion2b_s34b_b79k`, không fine-tune. Ảnh được tải song song qua URL (hoặc đọc từ thư mục `VG_100K*` nếu có dataset ảnh gắn vào),
mã hoá theo shard và cache trong `features/shards/` nên phiên bị ngắt vẫn chạy tiếp. Đặc trưng chuẩn hoá L2, lưu float16:
`features/clip_<model>.npz` = `img (Ni,512)`, `cap (Nc,512)`, `cap_img (Nc,)`, `ok (Ni,)` (ảnh tải hỏng bị loại khỏi mọi gallery).

In [ ]:
FEAT_FILE = WORK / "features" / f"clip_{CFG['clip_model']}_{CFG['clip_pretrained'].split('_')[0]}.npz"

def find_local_images():
    """vg_id -> đường dẫn ảnh nếu có dataset ảnh VG trong /kaggle/input (VG_100K, VG_100K_2); trống = dùng URL."""
    found = {}
    for d in glob.glob("/kaggle/input/*/**/VG_100K*", recursive=True):
        if os.path.isdir(d):
            for e in os.scandir(d):
                if e.name.endswith(".jpg"):
                    found[e.name[:-4]] = e.path
    return found

def load_image(im, local):
    p = local.get(str(im["vg_id"]))
    if p:
        return Image.open(p).convert("RGB")
    last = None
    for url in (im["url"], im.get("coco_url")):
        for _ in range(3):
            try:
                r = SESSION.get(url, timeout=30); r.raise_for_status()
                return Image.open(io.BytesIO(r.content)).convert("RGB")
            except Exception as e:  # noqa: BLE001
                last = e
    raise RuntimeError(f"{im['id']}: {last}")

import io
SESSION = requests.Session()
SESSION.mount("https://", requests.adapters.HTTPAdapter(pool_connections=64, pool_maxsize=64))
SESSION.mount("http://", requests.adapters.HTTPAdapter(pool_connections=64, pool_maxsize=64))

def extract_features():
    if FEAT_FILE.exists():
        log(f"{FEAT_FILE.name} đã có, bỏ qua"); return
    model, _, preprocess = open_clip.create_model_and_transforms(CFG["clip_model"], pretrained=CFG["clip_pretrained"])
    model = model.to(DEV).eval(); tok = open_clip.get_tokenizer(CFG["clip_model"])
    local = find_local_images(); log(f"ảnh cục bộ: {len(local)} (0 = stream qua URL)")
    shard_dir = WORK / "features" / "shards"; shard_dir.mkdir(exist_ok=True)
    D, ss, bs = model.visual.output_dim, CFG["shard_size"], CFG["batch_size"]

    def load(im):
        try:
            return preprocess(load_image(im, local))
        except Exception:  # noqa: BLE001
            return None

    feats, oks = [], []
    with ThreadPoolExecutor(CFG["workers"]) as ex:
        for s in range(0, len(images), ss):
            fp, op = shard_dir / f"img_{s:07d}.npy", shard_dir / f"ok_{s:07d}.npy"
            if fp.exists() and op.exists():
                feats.append(np.load(fp)); oks.append(np.load(op)); continue
            shard = images[s:s + ss]
            F, ok = np.zeros((len(shard), D), np.float16), np.zeros(len(shard), bool)
            for b in range(0, len(shard), bs):
                tens = list(ex.map(load, shard[b:b + bs]))
                idx = [i for i, t in enumerate(tens) if t is not None]
                if not idx:
                    continue
                x = torch.stack([tens[i] for i in idx]).to(DEV)
                with torch.no_grad(), torch.autocast(DEV.type, enabled=DEV.type == "cuda"):
                    f = torch.nn.functional.normalize(model.encode_image(x).float(), dim=-1)
                F[[b + i for i in idx]] = f.cpu().numpy().astype(np.float16); ok[[b + i for i in idx]] = True
            np.save(fp, F); np.save(op, ok); feats.append(F); oks.append(ok)
            log(f"ảnh {s + len(shard)}/{len(images)} ({(~ok).sum()} lỗi trong shard)")
    img, ok = np.concatenate(feats), np.concatenate(oks)
    log(f"{ok.sum()}/{len(ok)} ảnh mã hoá thành công")

    texts, cap_img = [c["text"] for c in captions], np.array([c["img"] for c in captions], np.int32)
    cap = []
    with torch.no_grad(), torch.autocast(DEV.type, enabled=DEV.type == "cuda"):
        for i in range(0, len(texts), 512):
            f = model.encode_text(tok(texts[i:i + 512]).to(DEV)).float()
            cap.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy().astype(np.float16))
    cap = np.concatenate(cap)
    np.savez(FEAT_FILE, img=img, cap=cap, cap_img=cap_img, ok=ok)
    save_json(dict(model=CFG["clip_model"], pretrained=CFG["clip_pretrained"], dim=int(D), n_images=len(images),
                   n_images_ok=int(ok.sum()), n_captions=len(captions), normalized=True, dtype="float16"),
              WORK / "features" / "meta.json")
    log(f"lưu {FEAT_FILE} img{img.shape} cap{cap.shape}")

extract_features()
feats = np.load(FEAT_FILE)
print({k: (feats[k].shape, str(feats[k].dtype)) for k in feats.files}, "| ảnh ok:", int(feats["ok"].sum()))

## 5. Chỉ mục FAISS và top-K ứng viên (dùng chung cho mọi phương pháp)

`IndexFlatIP` trên đặc trưng ảnh đã chuẩn hoá = cosine chính xác (gallery ≤ vài nghìn ảnh nên không cần chỉ mục xấp xỉ).
`Protocol.candidates()` lưu cho mỗi split: `q_cap, q_img, cand (Nq,K), sim (Nq,K), pos, gt_rank, gallery` →
Huy/Khánh chỉ việc chấm điểm lại `cand` rồi gọi `P.evaluate`. Train chia thành gallery 1K ảnh (giống test) và ảnh đúng luôn nằm trong top-K
(thay ứng viên cuối) để làm hard negative khi huấn luyện GNN.

In [ ]:
P = Protocol.load(WORK, feature_file=FEAT_FILE)

# FAISS index cho gallery test (toàn bộ pool) + ví dụ truy vấn tự do
gal_test = P.gallery("test", seed=None)
index = faiss.IndexFlatIP(P.img.shape[1]); index.add(np.ascontiguousarray(P.img[gal_test]))
faiss.write_index(index, str(WORK / "features" / "faiss_test.index"))
save_json(dict(gallery_image_idx=gal_test), WORK / "features" / "faiss_test_gallery.json")

def clip_text_query(texts):
    model, _, _ = open_clip.create_model_and_transforms(CFG["clip_model"], pretrained=CFG["clip_pretrained"])
    tok = open_clip.get_tokenizer(CFG["clip_model"]); model = model.to(DEV).eval()
    with torch.no_grad():
        f = torch.nn.functional.normalize(model.encode_text(tok(texts).to(DEV)).float(), dim=-1)
    return f.cpu().numpy()

q = clip_text_query(["a child playing football", "a giraffe standing next to a tree"])
s, i = index.search(q, 5)
for qt, ss, ii in zip(["a child playing football", "a giraffe standing next to a tree"], s, i):
    print(f"\n'{qt}' -> top-5 (FAISS, gallery test {len(gal_test)} ảnh)")
    for sc, j in zip(ss, ii):
        im = images[gal_test[j]]; print(f"   cos={sc:.3f}  {im['id']:>12s}  objects={im['objects'][:8]}")

# Candidates: test / val (toàn pool + từng fold seed) và train (gallery 1K, ảnh đúng luôn có mặt)
for split in ("test", "val"):
    for seed in [None] + CFG["seeds"]:
        c = P.candidates(split, seed)
        tag = "full" if seed is None else f"seed{seed}"
        np.savez(WORK / "candidates" / f"{split}_{tag}.npz", **{k: v for k, v in c.items() if isinstance(v, np.ndarray)},
                 gallery=np.array(c["gallery"]))
        log(f"{split:>5s} {tag:>5s}: {len(c['q_cap'])} truy vấn, gallery {c['gallery_size']}, ảnh đúng trong top-{P.K}: {(c['pos'] >= 0).mean():.3f}")
c = P.candidates("train", ensure_positive=True, captions_per_image=CFG["captions_per_image_train"])
np.savez(WORK / "candidates" / "train_galleries.npz", **{k: v for k, v in c.items() if isinstance(v, np.ndarray)}, gallery=np.array(c["gallery"]))
log(f"train: {len(c['q_cap'])} truy vấn trong {len(c['gallery']) // max(1, c['gallery_size'])} gallery x {c['gallery_size']} ảnh")

## 6. Baseline CLIP thuần — 3 seed, mean ± std

CLIP đóng băng là **tất định**: cùng truy vấn, cùng gallery thì cùng kết quả. Vì vậy giao thức seed của nhóm được định nghĩa như sau
(và các mô hình học của Huy/Khánh dùng **đúng các fold này** cộng thêm seed khởi tạo mô hình):

- **Fold theo seed** `s ∈ {0,1,2}`: gallery = 1 000 ảnh lấy ngẫu nhiên (`RandomState(s)`) từ pool test (2 138 ảnh); truy vấn = toàn bộ 5 caption/ảnh (≈5 000 truy vấn).
  Đây là giao thức *test-1K* quen thuộc của COCO; mean ± std cho thấy độ nhạy với cách chọn gallery.
- **Full**: gallery = toàn bộ pool test (khó hơn vì nhiều ảnh hơn) — số tất định, dùng làm số chính của báo cáo và để kiểm định ghép cặp.

Cùng giao thức trên `val` để Huy chọn α / epoch, và chiều **image → text** (nhánh B).

In [ ]:
clip_score = lambda c, p: c["sim"]          # baseline: điểm = cosine CLIP, không xếp lại
NAME = f"CLIP {CFG['clip_model']} ({CFG['clip_pretrained'].split('_')[0]})"
results = {}
results["t2i_test"] = P.evaluate(clip_score, NAME, split="test", seeds=CFG["seeds"])
results["t2i_val"] = P.evaluate(clip_score, NAME, split="val", seeds=CFG["seeds"])
results["i2t_test"] = P.evaluate_i2t(NAME, split="test", seeds=CFG["seeds"])
results["i2t_val"] = P.evaluate_i2t(NAME, split="val", seeds=CFG["seeds"])

def strip_ranks(res):
    """Bản để lưu JSON (hạng từng truy vấn lưu riêng trong .npz)."""
    r = json.loads(json.dumps(res, default=lambda o: o.tolist() if isinstance(o, np.ndarray) else float(o)))
    for ps in r["per_seed"]:
        ps.pop("ranks", None)
    return r

for k, r in results.items():
    np.savez(WORK / "results" / f"ranks_clip_{k}.npz", **{("full" if ps["seed"] is None else f"seed{ps['seed']}"): np.array(ps["ranks"]) for ps in r["per_seed"]})
save_json({k: strip_ranks(r) for k, r in results.items()} | {"config": CFG, "split_meta": P.split["meta"]}, WORK / "results" / "clip_baseline.json")

from IPython.display import Markdown, display
md_lines = [f"# Baseline CLIP thuần — VG ∩ COCO (split seed {CFG['seed_data']}, gallery {CFG['gallery_size']}, K={CFG['top_k']})", "",
            summary_table([results["t2i_test"], results["t2i_val"], results["i2t_test"], results["i2t_val"]],
                          title=f"Mean ± std trên {len(CFG['seeds'])} fold gallery theo seed {CFG['seeds']}"), "",
            summary_table([results["t2i_test"], results["t2i_val"], results["i2t_test"], results["i2t_val"]], field="full",
                          title="Toàn bộ pool (tất định)"), ""]
rows = []
for split_name, r in (("test", results["t2i_test"]), ("val", results["t2i_val"])):
    for g, agg in r["by_group"].items():
        rows.append(dict(name=f"{NAME} — {g}", split=split_name, mean_std=agg))
md_lines += [summary_table(rows, title="Text → image theo nhóm truy vấn (mean ± std theo seed)")]
(WORK / "results" / "clip_baseline.md").write_text("\n".join(md_lines), encoding="utf-8")
display(Markdown("\n".join(md_lines)))

### Biểu đồ cho báo cáo

In [ ]:
import matplotlib.pyplot as plt
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]   # thứ tự cố định: fold0, fold1, fold2 / direct, indirect, no_concept
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "axes.axisbelow": True, "figure.dpi": 110})
FIG = WORK / "results" / "figures"; FIG.mkdir(exist_ok=True)
r_test = results["t2i_test"]

# (a) R@k & MRR theo nhóm truy vấn — nhóm nào CLIP yếu là nơi đồ thị (Huy) cần cải thiện
groups = [g for g in ("direct", "indirect", "no_concept") if g in r_test["by_group"]]
ms = ["R@1", "R@5", "R@10", "MRR"]
fig, ax = plt.subplots(figsize=(7, 3.4))
w = 0.8 / len(groups)
for gi, g in enumerate(groups):
    agg = r_test["by_group"][g]
    x = np.arange(len(ms)) + (gi - (len(groups) - 1) / 2) * w
    ax.bar(x, [agg[m]["mean"] for m in ms], w * 0.92, yerr=[agg[m]["std"] for m in ms], color=PAL[gi],
           label=f"{g} (n≈{agg['n']})", capsize=2, linewidth=0)
    for xi, m in zip(x, ms):
        ax.text(xi, agg[m]["mean"] + agg[m]["std"] + 1.5, f"{agg[m]['mean']:.1f}", ha="center", fontsize=7, color="#52514e")
ax.set_xticks(range(len(ms))); ax.set_xticklabels(ms); ax.set_ylabel("%"); ax.set_ylim(0, 112); ax.set_yticks(range(0, 101, 20))
ax.set_title(f"CLIP thuần, text → image, test: gallery {r_test['gallery_size']}, mean ± std {len(CFG['seeds'])} fold, theo nhóm truy vấn", fontsize=9.5)
ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
fig.tight_layout(); fig.savefig(FIG / "clip_by_group.png", bbox_inches="tight"); plt.show()

# (b) phân bố hạng ảnh đúng (CDF) — trục x log; đường càng cao càng tốt
fig, ax = plt.subplots(figsize=(6, 3.2))
for si, ps in enumerate(r_test["per_seed"]):
    rk = np.sort(np.array(ps["ranks"]))
    lab = "toàn bộ pool" if ps["seed"] is None else f"fold seed {ps['seed']}"
    ax.plot(rk, np.arange(1, len(rk) + 1) / len(rk) * 100, color=PAL[si] if ps["seed"] is not None else "#52514e",
            lw=2 if ps["seed"] is None else 1.5, ls="-" if ps["seed"] is not None else "--", label=lab)
ax.set_xscale("log"); ax.set_xlabel("hạng của ảnh đúng (log)"); ax.set_ylabel("% truy vấn có hạng ≤ x")
ax.axvline(10, color="#999", lw=.8, ls=":"); ax.set_title("CLIP thuần: CDF hạng ảnh đúng (text → image, test)", fontsize=10)
ax.legend(frameon=False, fontsize=8); fig.tight_layout(); fig.savefig(FIG / "clip_rank_cdf.png"); plt.show()

## 7. Cách cắm một phương pháp khác vào module (ví dụ minh hoạ cho Huy / Khánh)

Một phương pháp = hàm `score_fn(cands, P) -> (Nq, K)`. Dưới đây là **ví dụ tối giản** `s_G` = tỉ lệ khái niệm trong truy vấn
có mặt trong scene graph của ảnh ứng viên (chỉ khớp tên, **không** dùng quan hệ, **không** phải mô hình đồ thị của nhóm — Huy thay bằng điểm GNN).
Quy trình đúng chuẩn chống rò rỉ: **α chọn trên val** (`tune_alpha`), rồi mới chạy test; so sánh với CLIP bằng **paired bootstrap** trên cùng truy vấn.

In [ ]:
obj_sets = [set(im["objects"]) for im in images]
cap_concepts = [set(c.get("concepts", [])) for c in captions]

def lexical_graph_score(c, P):
    out = np.zeros(c["sim"].shape, np.float64)
    for r, qc in enumerate(c["q_cap"]):
        cs = cap_concepts[qc]
        if cs:
            out[r] = [len(cs & obj_sets[ii]) / len(cs) for ii in c["cand"][r]]
    return out

# 1) chọn α trên VAL (toàn pool)
c_val = P.candidates("val", None)
alpha, curve = tune_alpha(c_val, lexical_graph_score(c_val, P), step=0.05, key="MRR")
print(f"α* (val, theo MRR) = {alpha}")
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.plot([d["alpha"] for d in curve], [d["MRR"] for d in curve], color=PAL[0], lw=2, label="MRR")
ax.plot([d["alpha"] for d in curve], [d["R@1"] for d in curve], color=PAL[1], lw=2, label="R@1")
ax.axvline(alpha, color="#52514e", ls="--", lw=1); ax.set_xlabel("α (1 = CLIP thuần)"); ax.set_ylabel("%")
ax.set_title("Độ nhạy theo α trên val — ví dụ CLIP + khớp tên đối tượng", fontsize=10); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIG / "alpha_sensitivity_example.png"); plt.show()

# 2) đánh giá trên TEST với α đã chốt — cùng fold, cùng top-K với baseline
example = P.evaluate(lambda c, p: fuse(c["sim"], lexical_graph_score(c, p), alpha), f"CLIP + khớp tên đối tượng (α={alpha})", "test", CFG["seeds"])
display(Markdown(summary_table([results["t2i_test"], example], title="Cùng giao thức: baseline vs ví dụ rerank (test, mean ± std)")))
rows = [dict(name=f"{r['name']} — {g}", split="test", mean_std=r["by_group"][g]) for r in (results["t2i_test"], example) for g in groups]
display(Markdown(summary_table(rows, title="Theo nhóm truy vấn")))

# 3) kiểm định ghép cặp trên toàn bộ pool test
boot = paired_bootstrap(results["t2i_test"]["full"]["ranks"], example["full"]["ranks"])
print(json.dumps(boot, indent=1))
save_json(dict(alpha=alpha, curve=curve, result=strip_ranks(example), bootstrap_vs_clip=boot), WORK / "results" / "example_rerank_lexical.json")

## 8. Phân tích lỗi của baseline (5 truy vấn thành công / 5 thất bại)

Lấy trên toàn bộ pool test: *thất bại* = ảnh đúng có hạng xấu nhất; *thành công* = hạng 1 với khoảng cách cosine lớn nhất so với ảnh hạng 2 (mỗi ảnh chỉ lấy một caption).
Bảng ghi nhóm truy vấn, khái niệm liên kết được và đối tượng trong scene graph của ảnh đúng — đầu vào cho phần "5 thành công / 5 thất bại" của báo cáo
và gợi ý loại truy vấn mà đồ thị cần cứu.

In [ ]:
from IPython.display import HTML
c_full = P.candidates("test", None)
ranks_full = np.array(results["t2i_test"]["full"]["ranks"])
margin = c_full["sim"][:, 0] - c_full["sim"][:, 1]
def pick(order, cond, k=5):   # mỗi ảnh đúng chỉ lấy 1 lần để 5 ví dụ là 5 ảnh khác nhau
    out, seen = [], set()
    for i in order:
        g = int(c_full["q_img"][i])
        if cond(i) and g not in seen:
            out.append(int(i)); seen.add(g)
        if len(out) == k:
            break
    return out
fail = pick(np.argsort(-ranks_full), lambda i: True)
succ = pick(np.argsort(-margin), lambda i: ranks_full[i] == 1)

def row(i, kind):
    qc, gt = int(c_full["q_cap"][i]), int(c_full["q_img"][i])
    return dict(kind=kind, rank=int(ranks_full[i]), group=captions[qc]["group"], query=captions[qc]["text"],
                query_concepts=", ".join(captions[qc].get("concepts", [])), gt_image=images[gt]["id"],
                gt_objects=", ".join(images[gt]["objects"][:12]),
                top1_image=images[int(c_full["cand"][i, 0])]["id"], top1_objects=", ".join(images[int(c_full["cand"][i, 0])]["objects"][:10]),
                gt_url=images[gt]["url"], top1_url=images[int(c_full["cand"][i, 0])]["url"])

rows = [row(i, "thất bại") for i in fail] + [row(i, "thành công") for i in succ]
df_err = pd.DataFrame(rows)
df_err.drop(columns=["gt_url", "top1_url"]).to_csv(WORK / "results" / "clip_success_failure.csv", index=False)
display(df_err.drop(columns=["gt_url", "top1_url"]))
html = "<table style='font-size:12px'><tr><th></th><th>truy vấn</th><th>ảnh đúng (hạng)</th><th>top-1 của CLIP</th></tr>"
for r in rows:
    html += (f"<tr><td>{r['kind']}</td><td style='max-width:260px'>{r['query']}<br><i>{r['group']}: {r['query_concepts']}</i></td>"
             f"<td><img src='{r['gt_url']}' width=160><br>{r['gt_image']} (hạng {r['rank']})<br><small>{r['gt_objects']}</small></td>"
             f"<td><img src='{r['top1_url']}' width=160><br>{r['top1_image']}<br><small>{r['top1_objects']}</small></td></tr>")
html += "</table>"
(WORK / "results" / "clip_success_failure.html").write_text(html, encoding="utf-8")
display(HTML(html))

## 9. Đóng gói bàn giao

| File | Ai dùng | Ghi chú |
|---|---|---|
| `split/split.json` | cả nhóm (Yến commit vào repo) | split cố định + meta (seed, pool, gallery, K) |
| `data/images.jsonl`, `data/captions.jsonl`, `data/object_vocab_train.json` | Huy (đồ thị), Khánh (truy vấn), Dương (giải thích) | objects/relations đã chuẩn hoá, nhóm truy vấn |
| `features/clip_*.npz`, `features/meta.json`, `features/faiss_test.index` | Huy, Khánh | đặc trưng CLIP đóng băng, float16, đã chuẩn hoá |
| `candidates/*.npz` | Huy, Khánh | top-K ứng viên CLIP cho test/val (full + 3 fold) và train (gallery 1K, có hard negative) |
| `mmxai_eval.py` | cả nhóm | module đánh giá dùng chung |
| `results/clip_baseline.{json,md}`, `results/figures/*.png`, `results/clip_success_failure.*` | Yến (báo cáo, slide) | bảng baseline, biểu đồ, phân tích lỗi |

Trên Kaggle, file zip nằm ở *Output* của notebook (không chứa shard ảnh để nhẹ).

In [ ]:
bundle = WORK.parent / "task1_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in WORK.rglob("*"):
        if p.is_file() and "shards" not in p.parts:
            z.write(p, p.relative_to(WORK))
    z.write("mmxai_eval.py", "mmxai_eval.py")
print(f"{bundle} — {bundle.stat().st_size / 1e6:.1f} MB")
print("\n".join(str(p.relative_to(WORK)) for p in sorted(WORK.rglob("*")) if p.is_file() and "shards" not in p.parts))